In [ ]:
words = open('../data/names.txt','r').read().splitlines()
string = sorted(list(set((''.join(words)))))

In [39]:
import torch 

stoi = { s: i + 1 for (i,s) in enumerate(string)}
itos = { i+1 : s for (i,s) in enumerate(string)}
stoi["."] = 0 
itos[0] = '.'
print(itos)
N = torch.zeros((27,27), dtype=torch.int32)

for word in words : 
    ch = ['.'] + list(word) + ['.']
    for ch1, ch2 in zip(ch, ch[1:]) : 
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] +=1 

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [62]:
g = torch.Generator().manual_seed(11)
p = N[0].float()
p = p/sum(p)

words = []
for i in range(10) : 
    l = 0
    word = ''
    while True : 
        p = N[l].float()
        p = p/sum(p)
        l = torch.multinomial(p, 1, replacement = True, generator=g).item()
        word += itos[l]
        if l == 0 : 
            break
    words.append(word)


words

['oaiqudeuerivi.',
 'ar.',
 'arhoyllalanin.',
 'toraiuennacenalexeyas.',
 'may.',
 'priemoela.',
 'amori.',
 'rseihyunifeyarijalyatranyn.',
 'h.',
 'jelardiaikeemon.']

In [97]:
#Create the training set of the bigram 

import random
random.seed(42)          # fixed seed → reproducible split
random.shuffle(words)


n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))



def build_dataset(words) : 

    xs , ys = [], []
    for word in words : 
        ch = ['.'] + list(word) + ['.']
        for ch1, ch2 in zip(ch, ch[1:]) : 
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            xs.append(ix1)
            ys.append(ix2)

    return torch.tensor(xs), torch.tensor(ys)
    

Xtr,  Ytr  = build_dataset(words[:n1])    # 80% — fit W
Xdev, Ydev = build_dataset(words[n1:n2])  # 10% — tune hyperparameters
Xte,  Yte  = build_dataset(words[n2:])    # 10% — touch ONCE, at the very end


W = torch.randn((27,27), requires_grad=True) # weights of the MLP 


In [98]:
import torch.nn.functional as F


def split_loss(X,Y) : 
    xenc = F.one_hot(X ,num_classes=27).float()
    logits = xenc@W
    return F.cross_entropy(logits, Y)


#gradient descent : 
for i in range(100) : 
    loss = split_loss(Xtr, Ytr)
    print('train', loss)

    W.grad = None
    loss.backward()

    #gradient descent
    W.data += -10*W.grad

    



train tensor(3.7180, grad_fn=<NllLossBackward0>)
train tensor(3.6253, grad_fn=<NllLossBackward0>)
train tensor(3.5455, grad_fn=<NllLossBackward0>)
train tensor(3.4750, grad_fn=<NllLossBackward0>)
train tensor(3.4118, grad_fn=<NllLossBackward0>)
train tensor(3.3546, grad_fn=<NllLossBackward0>)
train tensor(3.3027, grad_fn=<NllLossBackward0>)
train tensor(3.2556, grad_fn=<NllLossBackward0>)
train tensor(3.2129, grad_fn=<NllLossBackward0>)
train tensor(3.1740, grad_fn=<NllLossBackward0>)
train tensor(3.1387, grad_fn=<NllLossBackward0>)
train tensor(3.1065, grad_fn=<NllLossBackward0>)
train tensor(3.0770, grad_fn=<NllLossBackward0>)
train tensor(3.0498, grad_fn=<NllLossBackward0>)
train tensor(3.0247, grad_fn=<NllLossBackward0>)
train tensor(3.0013, grad_fn=<NllLossBackward0>)
train tensor(2.9796, grad_fn=<NllLossBackward0>)
train tensor(2.9593, grad_fn=<NllLossBackward0>)
train tensor(2.9402, grad_fn=<NllLossBackward0>)
train tensor(2.9223, grad_fn=<NllLossBackward0>)
train tensor(2.9055,

In [101]:
print('dev  ', split_loss(Xdev, Ydev))


dev   tensor(2.5609, grad_fn=<NllLossBackward0>)
